# Resume tail grid (uqfusion Phase 1)

Restarts the 9-variant / 3-seed tail after `yolo12l seed 2`. Grid is resume-safe: rows already in the CSV are skipped.

Run from repo root (`/workspace/Saroha_Work`). Flags below must stay identical to the first 12 runs (`--classes 0`, same `--data`, same `--out-csv`) or the grid refuses to resume.

## 1. Nothing still running
Empty output = clear. Any PID = kill it in the next cell.

In [ ]:
!pgrep -af run_benchmark.py || echo "clear"

In [ ]:
# only if step 1 printed a PID
# !pkill -f run_benchmark.py

## 2. What the CSV counts as done
Expect 12 data rows: 12s / 12m / 12l x seeds 0,1,2.

In [ ]:
!cut -d, -f1,2 runs/benchmark/benchmark_results_tail.csv | column -t -s,
!echo '--- data rows:' $(( $(wc -l < runs/benchmark/benchmark_results_tail.csv) - 1 ))

## 3. Move aside the interrupted run dir
Otherwise Ultralytics auto-increments to `..._seed02` and dir names stop matching the CSV.

In [ ]:
!d=runs/benchmark/runs/ship_yolo12x_seed0; [ -d $d ] && mv $d runs/benchmark/runs/_partial_ship_yolo12x_seed0 && echo moved || echo 'no partial dir'

## 4. Run
Output streams into this cell and appends to `runs/grid_tail.log`. **Kernel disconnect kills the job** — for a detach-safe run use tmux in a terminal instead (`tmux new -s grid`, paste the same command without `!`, detach `Ctrl-b d`).

In [ ]:
!python -u scripts/run_benchmark.py \
    --data runs/derived/data_vis_stride2.yaml \
    --classes 0 \
    --seeds 0 1 2 \
    --variants yolo12s yolo12m yolo12l yolo12x yolo26n yolo26s yolo26m yolo26l yolo26x \
    --out-csv runs/benchmark/benchmark_results_tail.csv \
    --run-prefix ship 2>&1 | tee -a runs/grid_tail.log

## 5. Check
Expect 30 data rows when the tail finishes.

In [ ]:
!echo 'data rows:' $(( $(wc -l < runs/benchmark/benchmark_results_tail.csv) - 1 )) '/ 30'
!column -t -s, runs/benchmark/benchmark_results_tail.csv